In [2]:
from selenium import webdriver
# from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
# from selenium.webdriver.chrome.options import Options
import chromedriver_autoinstaller
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from pydantic import BaseModel, AnyUrl, ValidationError
import json


chromedriver_autoinstaller.install()  # Instala o ChromeDriver automaticamente


'c:\\Users\\user\\AppData\\Local\\Programs\\Python\\Python312\\Lib\\site-packages\\chromedriver_autoinstaller\\140\\chromedriver.exe'

In [3]:
class Imovel(BaseModel):
    title: str
    street: str
    link: str
    price: float
    location: str
    rooms: int
    area: float
    bathrooms: int
    parking: int

In [3]:
def get_important_data_zapimoveis(element):
    # Link do imóvel
    link = element.find_element(By.TAG_NAME, "a").get_attribute("href")
    if link is None:
        link = "None"
    
    # Título do imóvel (ex: "Sala/Conjunto para alugar com 95 m², 1 banheiro, 1 vaga em Jardim, Santo André")
    title = element.find_element(By.TAG_NAME, "a").get_attribute("title")
    
    # Localização (bairro/cidade)
    location = element.find_element(By.CSS_SELECTOR, '[data-cy="rp-cardProperty-location-txt"]').text.strip()
    location = location.split("\n")[1] if "\n" in location else location
    
    # Endereço (rua)
    try:
        street = element.find_element(By.CSS_SELECTOR, '[data-cy="rp-cardProperty-street-txt"]').text.strip()
    except:
        street = ""
    
    # Área
    try:
        area_txt = element.find_element(By.CSS_SELECTOR, '[data-cy="rp-cardProperty-propertyArea-txt"]').text
        area = float(area_txt.split("\n")[-1].replace("m²", "").replace(",", ".").strip())
    except:
        area = 0.0

    # Banheiros
    try:
        bathrooms_txt = element.find_element(By.CSS_SELECTOR, '[data-cy="rp-cardProperty-bathroomQuantity-txt"]').text
        bathrooms = int(bathrooms_txt.split("\n")[-1])
    except:
        bathrooms = 0

    # Vagas
    try:
        rooms_txt = element.find_element(By.CSS_SELECTOR, '[data-cy="rp-cardProperty-bedroomQuantity-txt"]').text
        rooms = int(rooms_txt.split("\n")[-1])
    except:
        rooms = 0

    # Preço
    try:
        price_txt = element.find_element(By.CSS_SELECTOR, '[data-cy="rp-cardProperty-price-txt"] p').text
        price = float(price_txt.split("\n")[-1].replace("R$", "").replace(".", "").replace(",", ".").split("/")[0])
    except:
        price = 0.0
 
    try:
        parking_txt = element.find_element(By.CSS_SELECTOR, '[data-cy="rp-cardProperty-parkingSpacesQuantity-txt"]').text
        parking = int(parking_txt.split("\n")[-1])
    except:
        parking = 0

    try:
        object_imovel = Imovel(
            title=title,
            street=street,
            link=link,
            price=price,
            location=location,
            rooms=rooms,
            area=area,
            bathrooms=bathrooms,
            parking=parking
        )
    except ValidationError as e:
        print(e.errors())
        object_imovel = None
    return object_imovel.model_dump()

In [47]:
# Zapimoveis
# Inicializa o navegador
driver = webdriver.Chrome()

# URL alvo
url = "https://www.zapimoveis.com.br/aluguel/"
driver.get(url)

lista_imoveis = driver.find_elements(By.CSS_SELECTOR, '[data-cy="rp-property-cd"]')
lista = [get_important_data_zapimoveis(links) for links in lista_imoveis]
driver.quit()  # Fecha o navegador

with open("zapimoveis.json", "w", encoding="utf-8") as file:
    json.dump(lista, file, ensure_ascii=False, indent=4)


In [30]:
def get_important_data_ImovelWeb(element):
    # Título do imóvel
    try:
        title = element.find_element(By.CSS_SELECTOR, '[data-qa="POSTING_CARD_DESCRIPTION"]').text
    except:
        title = "None"

    # Link do imóvel
    try:
        link = element.find_element(By.TAG_NAME, 'a').get_attribute("href")
    except:
        link = "None"

    # Localização (bairro/cidade)
    try:
        location = element.find_element(By.CSS_SELECTOR, '[data-qa="POSTING_CARD_LOCATION"]').text.strip()
    except:
        location = "None"

    # Endereço (rua)
    try:
        street = element.find_element(By.CLASS_NAME, "postingLocations-module__location-address-in-listing").text.strip()
    except:
        street = "None"

    # Área, quartos, banheiros, vagas
    area = 0.0
    rooms = 0
    bathrooms = 0
    parking = 0
    try:
        features = element.find_element(By.CSS_SELECTOR, '[data-qa="POSTING_CARD_FEATURES"]')
        spans = features.find_elements(By.TAG_NAME, "span")
        list_values = []
        for span in spans:
            list_values.append(span.text.split()[0])
        area, rooms, bathrooms, parking = list_values[:4]

    except:
        area = 0.0
        rooms = 0
        bathrooms = 0
        parking = 0

    # Preço
    try:
        price_txt = element.find_element(By.CSS_SELECTOR, '[data-qa="POSTING_CARD_PRICE"]').text
        price = float(price_txt.replace("R$", "").replace(".", "").replace(",", ".").split()[0])
    except:
        price = 0.0

    try:
        object_imovel = Imovel(
            title=title,
            street=street,
            link=link,
            price=price,
            location=location,
            rooms=rooms,
            area=area,
            bathrooms=bathrooms,
            parking=parking
        )
    except ValidationError as e:
        print(e.errors())
        object_imovel = None

    return object_imovel.model_dump() if object_imovel else None

In [ ]:
# ImovelWeb
# Inicializa o navegador
driver = webdriver.Chrome()

# URL alvo
url = "https://www.imovelweb.com.br/imoveis-venda-santa-catarina.html"
driver.get(url)
for _ in range(5):  # Número de páginas a serem navegadas
    lista_imoveis = driver.find_elements(By.CLASS_NAME, 'postingsList-module__card-container')
    lista = [get_important_data_ImovelWeb(links) for links in lista_imoveis]
    button = driver.find_element(By.CSS_SELECTOR, '[data-qa="PAGING_NEXT"]')
    if button:  # Verifica se o botão Next-Page existe
        button.click()
    else:
        break
driver.quit()  # Fecha o navegador

with open("imoveisweb.json", "w", encoding="utf-8") as file:
    json.dump(lista, file, ensure_ascii=False, indent=4)

In [7]:
def get_important_data_brognoli(element):
    # Título do imóvel
    try:
        title = element.find_element(By.TAG_NAME, 'a').get_attribute("title")
    except:
        title = "None"

    # Link do imóvel
    try:
        link = element.find_element(By.TAG_NAME, 'a').get_attribute("href")
    except:
        link = "None"

    # Localização (bairro/cidade)
    try:
        location = element.find_element(By.CSS_SELECTOR, 'span.e').text.strip()
        street = location.split(",")[0] if "," in location else location
        location = location.split(",")[1] if "," in location else location
    except:
        location = street = "None"

    # Área, quartos, banheiros, vagas
    area = 0.0
    rooms = 0
    bathrooms = 0
    parking = 0
    try:
        features = element.find_elements(By.TAG_NAME, 'li')
        for feature in features:
            text = feature.text.lower()
            if 'm²' in text:
                area = float(text.replace('m²', '').strip())
            elif 'quartos' in text or 'dormitório' in text:
                rooms = int(text.split()[0])
            elif 'banheiro' in text:
                bathrooms = int(text.split()[0])
            else:
                parking = int(text.strip())
    except:
        area = 0.0
        rooms = 0
        bathrooms = 0
        parking = 0

    # Preço
    try:
        price_txt = element.find_element(By.CSS_SELECTOR, 'span.v').text
        price = float(price_txt.replace("Valor:", "").replace("R$", "").replace(".", "").replace(",", ".").strip())
    except:
        price = 0.0

    try:
        object_imovel = Imovel(
            title=title,
            street=street,
            link=link,
            price=price,
            location=location,
            rooms=rooms,
            area=area,
            bathrooms=bathrooms,
            parking=parking
        )
    except ValidationError as e:
        print(e.errors())
        object_imovel = None

    return object_imovel.model_dump() if object_imovel else None

In [11]:
# Brognoli
# Inicializa o navegador
driver = webdriver.Chrome()

# URL alvo
url = "https://www.brognoli.com.br/comprar/cidade/biguacu/1"
driver.get(url)  # <article class="imovel">
lista = []
for _ in range(5):  # Exemplo: navegar por 5 páginas
    lista_imoveis = driver.find_elements(By.CSS_SELECTOR, 'article.imovel')
    lista += [get_important_data_brognoli(links) for links in lista_imoveis]
    try:
        active_page = driver.find_element(By.CSS_SELECTOR, 'ul.pagination li a.active')
        pagina_atual = int(active_page.text)  # Ou active_page.get_attribute("title")
        next_page_selector = f'ul.pagination li a[title="Página {pagina_atual+1}"]'
        next_page = driver.find_element(By.CSS_SELECTOR, next_page_selector)
        if next_page: # Verifica se o botão Next-Page existe
            driver.execute_script("arguments[0].click();", next_page)  # Usa JS para garantir o clique
            # Espera até que o documento esteja pronto
            WebDriverWait(driver, 10).until(
                lambda d: d.execute_script("return document.readyState") == "complete"
            )
        else:
            break
    except Exception as e:
        print(f"Erro ao clicar no botão: {e}")
        break
driver.quit()  # Fecha o navegador
with open("brognoli.json", "w", encoding="utf-8") as file:
    json.dump(lista, file, ensure_ascii=False, indent=4)

Erro ao clicar no botão: Message: no such element: Unable to locate element: {"method":"css selector","selector":"ul.pagination li a[title="Página 3"]"}
  (Session info: chrome=140.0.7339.187); For documentation on this error, please visit: https://www.selenium.dev/documentation/webdriver/troubleshooting/errors#nosuchelementexception
Stacktrace:
	GetHandleVerifier [0x0xdec333+65459]
	GetHandleVerifier [0x0xdec374+65524]
	(No symbol) [0x0xc0d973]
	(No symbol) [0x0xc576e7]
	(No symbol) [0x0xc57a8b]
	(No symbol) [0x0xc9dea2]
	(No symbol) [0x0xc79e44]
	(No symbol) [0x0xc9b606]
	(No symbol) [0x0xc79bf6]
	(No symbol) [0x0xc4b38e]
	(No symbol) [0x0xc4c274]
	GetHandleVerifier [0x0x106eda3+2697763]
	GetHandleVerifier [0x0x1069ec7+2677575]
	GetHandleVerifier [0x0xe14194+228884]
	GetHandleVerifier [0x0xe049f8+165496]
	GetHandleVerifier [0x0xe0b18d+192013]
	GetHandleVerifier [0x0xdf47d8+99416]
	GetHandleVerifier [0x0xdf4972+99826]
	GetHandleVerifier [0x0xddebea+10346]
	BaseThreadInitThunk [0x0x772